In [0]:
dbutils.fs.ls('/Volumes/mydata/mysc/myvol')

[FileInfo(path='dbfs:/Volumes/mydata/mysc/myvol/BigMart Sales.csv', name='BigMart Sales.csv', size=869537, modificationTime=1780503434000),
 FileInfo(path='dbfs:/Volumes/mydata/mysc/myvol/Coffe_sales.csv', name='Coffe_sales.csv', size=260602, modificationTime=1781107980000)]

## Bronze Layer 

#### Read CSV

In [0]:
bronze_df=spark.read.format("csv").option("inferschema",True).option("header",True).load("/Volumes/mydata/mysc/myvol/Coffe_sales.csv")

#### Store Raw Data

In [0]:
bronze_df.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/bronze/coffee_sales")

#### Importing Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

## Silver Layer

In [0]:
#Checking if there is any null row in the column
bronze_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
]).display()

hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,0,0,0,0,0,0,0,0,0,0


In [0]:
#Remove duplicate records
silver_df=bronze_df.dropDuplicates()

In [0]:
#Remove negative or zero sales
silver_df = silver_df.filter(col("money") > 0)

In [0]:
#validate Hours
silver_df=silver_df.filter((col("hour_of_day")>=0) & (col("hour_of_day")<=23))

In [0]:
#Convert Date Column
silver_df=silver_df.withColumn("Date",to_date(col("Date"),'yyyy-MM-dd'))

In [0]:
#Convert Time Column
silver_df = silver_df.withColumn(
    "Time",
    to_timestamp(col("Time"),"HH:mm:ss")
)

In [0]:
#Standardize Coffee Names
silver_df=silver_df.withColumn("coffee_name",initcap(trim(col("coffee_name"))))

In [0]:
#Standardize Payment Type
silver_df=silver_df.withColumn("cash_type",lower(trim(col("cash_type"))))

In [0]:
#Extract Year
silver_df=silver_df.withColumn("Year",year(col("Date")))

In [0]:
#Extract Quarter
silver_df=silver_df.withColumn("Quarter",quarter(col("Date")))

In [0]:
#Weekend Flag
silver_df=silver_df.withColumn("is_weekend",when(col("Weekday").isin("Sat","Sun"),1).otherwise(0))

In [0]:
#High Value Transaction
silver_df=silver_df.withColumn("High_value_sale",when(col("money")>(silver_df.agg({"money":"avg"}).collect()[0][0]),1).otherwise(0))

In [0]:
#Peak Hour
silver_df=silver_df.withColumn("Peak_Hour",when((col("hour_of_day")>=7)&(col("hour_of_day")<=10),1).otherwise(0))

In [0]:
#Daily Revenue
daily_sales=silver_df.groupBy("Date").agg(sum("money").alias("Daily_Revenue")).orderBy("Date")

In [0]:
#Monthly Revenue
monthly_sales=silver_df.groupBy("Month_name","Monthsort").agg(sum("money").alias("Monthly_Revenue")).orderBy("Monthsort")

In [0]:
silver_df=silver_df.withColumn("Spend_Tier",when((col('money')<3),"Low").when((col('money')<6),"Medium").otherwise("High"))

In [0]:
# Revenue by coffee name
coffee_sales=silver_df.groupBy("Coffee_Name").agg(sum("money").alias("Total_Revenue"),avg("money").alias("Average_Revenue"),count("*").alias("Number_Orders")).orderBy("Total_Revenue")
coffee_sales.display()

Coffee_Name,Total_Revenue,Average_Revenue,Number_Orders
Espresso,2690.2799999999947,20.854883720930193,129
Cortado,7384.860000000021,25.731219512195192,287
Cocoa,8521.160000000027,35.653389121339025,239
Hot Chocolate,9933.460000000025,35.99079710144937,276
Americano,14650.259999999897,25.975638297872155,564
Cappuccino,17439.139999999985,35.883004115226306,486
Americano With Milk,24751.119999999948,30.5947095179233,809
Latte,26875.29999999979,35.50237780713314,757


In [0]:
#Hourly Traffic
Hourly_Sale=silver_df.groupBy("hour_of_day").agg(count("*").alias("Number_of_Customers"),sum("money").alias("Revenue")).orderBy("hour_of_day")
Hourly_Sale.display()

hour_of_day,Number_of_Customers,Revenue
6,5,149.39999999999998
7,88,2846.020000000003
8,235,7017.880000000015
9,242,7264.280000000007
10,328,10198.52
11,283,8453.100000000013
12,241,7419.620000000009
13,225,7028.7600000000075
14,225,7173.800000000013
15,236,7476.020000000013


In [0]:
#Revenue By Payment Method
payment_sales = silver_df.groupBy("cash_type").agg(sum("money").alias("Total_Revenue"))
payment_sales.display()

cash_type,Total_Revenue
card,112245.5800000001


In [0]:
#Revenue by Time of Day
time_sales = silver_df.groupBy("Time_of_Day").agg(sum("money").alias("Total_Revenue"))
time_sales.display()

Time_of_Day,Total_Revenue
Night,38186.339999999815
Morning,35929.19999999977
Afternoon,38130.03999999973


In [0]:
#Top Selling Coffee
coffee_rank=silver_df.groupBy("Coffee_Name").agg(sum("money").alias("revenue")).withColumn("Rank",dense_rank().over(Window.orderBy(col("revenue").desc())))
coffee_rank.display()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Coffee_Name,revenue,Rank
Latte,26875.29999999979,1
Americano With Milk,24751.119999999948,2
Cappuccino,17439.139999999985,3
Americano,14650.259999999897,4
Hot Chocolate,9933.460000000025,5
Cocoa,8521.160000000027,6
Cortado,7384.860000000021,7
Espresso,2690.2799999999947,8


In [0]:
#Running Revenue
running_sales=daily_sales.withColumn("Running_Revenue",sum("Daily_Revenue").over(Window.orderBy("Date")))
running_sales.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Date,Daily_Revenue,Running_Revenue
2024-03-01,396.3,396.3
2024-03-02,188.1,584.4
2024-03-03,309.09999999999997,893.5
2024-03-04,135.2,1028.7
2024-03-05,338.5,1367.2
2024-03-06,135.2,1502.4
2024-03-07,140.1,1642.5
2024-03-08,265.50000000000006,1908.0
2024-03-09,439.3999999999999,2347.4
2024-03-10,91.6,2439.0


In [0]:
# Revenue Contribution %
total_rev=silver_df.agg(sum("money")).collect()[0][0]
coffee_sales=coffee_sales.withColumn("Revenue_Contribution",round(col("Total_Revenue")/lit(total_rev)*100,))
coffee_sales.display()

Coffee_Name,Total_Revenue,Average_Revenue,Number_Orders,Revenue_Contribution
Espresso,2690.2799999999947,20.854883720930193,129,2.0
Cortado,7384.860000000021,25.731219512195192,287,7.0
Cocoa,8521.160000000027,35.653389121339025,239,8.0
Hot Chocolate,9933.460000000025,35.99079710144937,276,9.0
Americano,14650.259999999897,25.975638297872155,564,13.0
Cappuccino,17439.139999999985,35.883004115226306,486,16.0
Americano With Milk,24751.119999999948,30.5947095179233,809,22.0
Latte,26875.29999999979,35.50237780713314,757,24.0


In [0]:
#KPIs
kpi_df=silver_df.agg(sum("money").alias("Total_Revenue"),avg("money").alias("Average_Revenue"),count("*").alias("Total_Orders"),countDistinct("coffee_name").alias("Unique_Coffees"))
kpi_df.display()

Total_Revenue,Average_Revenue,Total_Orders,Unique_Coffees
112245.57999999968,31.645215675218406,3547,8


### Load Phase

In [0]:
silver_df.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/silver/coffee_sales")

In [0]:
daily_sales.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/gold/daily_sales")

In [0]:
monthly_sales.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/gold/monthly_sales")

In [0]:
coffee_rank.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/gold/coffee_rank")

In [0]:
kpi_df.write.mode("overwrite").parquet("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/gold/kpi_df")

In [0]:
daily_sales.write.mode("overwrite").csv("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/CSV/daily_sales")
monthly_sales.write.mode("overwrite").csv("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/CSV/monthly_sales")
coffee_rank.write.mode("overwrite").csv("/Volumes/mydata/mysc/myvol/Coffee_Sales_ETL_Project/CSV/coffee_rank")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
silver_df.printSchema()

root
 |-- hour_of_day: integer (nullable = true)
 |-- cash_type: string (nullable = true)
 |-- money: double (nullable = true)
 |-- coffee_name: string (nullable = true)
 |-- Time_of_Day: string (nullable = true)
 |-- Weekday: string (nullable = true)
 |-- Month_name: string (nullable = true)
 |-- Weekdaysort: integer (nullable = true)
 |-- Monthsort: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- High_value_sale: integer (nullable = false)
 |-- Peak_Hour: integer (nullable = false)
 |-- Spend_Tier: string (nullable = false)

